# Problem Statement

## **Business Context**

"Visit with Us," a leading travel company, is revolutionizing the tourism industry by leveraging data-driven strategies to optimize operations and customer engagement. While introducing a new package offering, such as the Wellness Tourism Package, the company faces challenges in targeting the right customers efficiently. The manual approach to identifying potential customers is inconsistent, time-consuming, and prone to errors, leading to missed opportunities and suboptimal campaign performance.

To address these issues, the company aims to implement a scalable and automated system that integrates customer data, predicts potential buyers, and enhances decision-making for marketing strategies. By utilizing an MLOps pipeline, the company seeks to achieve seamless integration of data preprocessing, model development, deployment, and CI/CD practices for continuous improvement. This system will ensure efficient targeting of customers, timely updates to the predictive model, and adaptation to evolving customer behaviors, ultimately driving growth and customer satisfaction.


## **Objective**

As an MLOps Engineer at "Visit with Us," your responsibility is to design and deploy an MLOps pipeline on GitHub to automate the end-to-end workflow for predicting customer purchases. The primary objective is to build a model that predicts whether a customer will purchase the newly introduced Wellness Tourism Package before contacting them. The pipeline will include data cleaning, preprocessing, transformation, model building, training, evaluation, and deployment, ensuring consistent performance and scalability. By leveraging GitHub Actions for CI/CD integration, the system will enable automated updates, streamline model deployment, and improve operational efficiency. This robust predictive solution will empower policymakers to make data-driven decisions, enhance marketing strategies, and effectively target potential customers, thereby driving customer acquisition and business growth.

## **Data Description**

The dataset contains customer and interaction data that serve as key attributes for predicting the likelihood of purchasing the Wellness Tourism Package. The detailed attributes are:

**Customer Details**
- **CustomerID:** Unique identifier for each customer.
- **ProdTaken:** Target variable indicating whether the customer has purchased a package (0: No, 1: Yes).
- **Age:** Age of the customer.
- **TypeofContact:** The method by which the customer was contacted (Company Invited or Self Inquiry).
- **CityTier:** The city category based on development, population, and living standards (Tier 1 > Tier 2 > Tier 3).
- **Occupation:** Customer's occupation (e.g., Salaried, Freelancer).
- **Gender:** Gender of the customer (Male, Female).
- **NumberOfPersonVisiting:** Total number of people accompanying the customer on the trip.
- **PreferredPropertyStar:** Preferred hotel rating by the customer.
- **MaritalStatus:** Marital status of the customer (Single, Married, Divorced).
- **NumberOfTrips:** Average number of trips the customer takes annually.
- **Passport:** Whether the customer holds a valid passport (0: No, 1: Yes).
- **OwnCar:** Whether the customer owns a car (0: No, 1: Yes).
- **NumberOfChildrenVisiting:** Number of children below age 5 accompanying the customer.
- **Designation:** Customer's designation in their current organization.
- **MonthlyIncome:** Gross monthly income of the customer.

**Customer Interaction Data**
- **PitchSatisfactionScore:** Score indicating the customer's satisfaction with the sales pitch.
- **ProductPitched:** The type of product pitched to the customer.
- **NumberOfFollowups:** Total number of follow-ups by the salesperson after the sales pitch.-
- **DurationOfPitch:** Duration of the sales pitch delivered to the customer.


# Model Building

In [1]:
# Create a master folder to keep all files created when executing the below code cells
import os
os.makedirs("tourism_project", exist_ok=True)

In [2]:
# Create a folder for storing the model building files
os.makedirs("tourism_project/model_building", exist_ok=True)

###  Criteria 1
### Data Registration

- Create a master folder and create a subfolder "data"
- Register the data on the Hugging Face dataset space

In [3]:
os.makedirs("tourism_project/data", exist_ok=True)

In [4]:
%%writefile tourism_project/model_building/data_register.py
from huggingface_hub.utils import RepositoryNotFoundError, HfHubHTTPError
from huggingface_hub import HfApi, create_repo
import os


repo_id = "nikhileshmehta89/tourism_package_prediction"
repo_type = "dataset"

# Initialize API client
api = HfApi(token=os.getenv("HF_TOKEN"))

# Step 1: Check if the space exists
try:
    api.repo_info(repo_id=repo_id, repo_type=repo_type)
    print(f"Space '{repo_id}' already exists. Using it.")
except RepositoryNotFoundError:
    print(f"Space '{repo_id}' not found. Creating new space...")
    create_repo(repo_id=repo_id, repo_type=repo_type, private=False)
    print(f"Space '{repo_id}' created.")

api.upload_folder(
    folder_path="tourism_project/data",
    repo_id=repo_id,
    repo_type=repo_type,
)

Writing tourism_project/model_building/data_register.py


Once the **data** folder created after executing the above cell, please upload the **tourism.csv** in to the folder

###  Criteria 2
### Data Preparation

- Load the dataset directly from the Hugging Face data space.
- Perform data cleaning and remove any unnecessary columns.
- Split the cleaned dataset into training and testing sets, and save them locally.
- Upload the resulting train and test datasets back to the Hugging Face data space.

In [ ]:
%%writefile tourism_project/model_building/prep.py
# for data manipulation
import pandas as pd
import sklearn
# for creating a folder
import os
# for data preprocessing and pipeline creation
from sklearn.model_selection import train_test_split
# for converting text data in to numerical representation
from sklearn.preprocessing import LabelEncoder
# for hugging face space authentication to upload files
from huggingface_hub import login, HfApi

HF_TOKEN = os.getenv("HF_TOKEN")
if not HF_TOKEN:
    raise ValueError("HF_TOKEN is not set")
api = HfApi(token=HF_TOKEN)

repo_id = "nikhileshmehta89/tourism_package_prediction"
repo_type = "dataset"

try:
    api.repo_info(repo_id=repo_id, repo_type=repo_type)
    print("Dataset repo already exists.")
except RepositoryNotFoundError:
    print("Dataset repo not found. Creating...")
    api.create_repo(
        repo_id=repo_id,
        repo_type=repo_type,
        private=False,
        exist_ok=True,
    )
    print("Dataset repo created.")

# Define constants for the dataset and output paths
DATASET_PATH = "tourism_project/data/tourism_package.csv"
df = pd.read_csv(DATASET_PATH)
print("Dataset loaded successfully.")

# ── Data Cleaning ──────────────────────────────────────────────────────────────
# Drop the unnamed index column and CustomerID (not useful for prediction)
df.drop(columns=["Unnamed: 0", "CustomerID"], inplace=True, errors="ignore")

# Fill missing values: median for numeric, mode for categorical
num_cols = df.select_dtypes(include="number").columns.tolist()
cat_cols = df.select_dtypes(include="object").columns.tolist()

for col in num_cols:
    df[col].fillna(df[col].median(), inplace=True)

for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

print("Missing values handled.")

# Encode categorical columns using LabelEncoder
le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col])

print("Categorical columns encoded.")

# ── Train / Test Split ─────────────────────────────────────────────────────────

X = df.drop(columns=["ProdTaken"])
y = df["ProdTaken"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

train_df = X_train.copy()
train_df["ProdTaken"] = y_train.values

test_df = X_test.copy()
test_df["ProdTaken"] = y_test.values

print(f"Train size: {len(train_df)}, Test size: {len(test_df)}")

# ── Save Locally ───────────────────────────────────────────────────────────────

os.makedirs("tourism_project/data", exist_ok=True)
train_df.to_csv("tourism_project/data/train.csv", index=False)
test_df.to_csv("tourism_project/data/test.csv", index=False)
print("Train and test datasets saved locally.")

# ── Upload to Hugging Face ─────────────────────────────────────────────────────

repo_id = "nikhileshmehta89/tourism_package_prediction"

api.upload_file(
    path_or_fileobj="tourism_project/data/train.csv",
    path_in_repo="train.csv",
    repo_id=repo_id,
    repo_type="dataset",
)

api.upload_file(
    path_or_fileobj="tourism_project/data/test.csv",
    path_in_repo="test.csv",
    repo_id=repo_id,
    repo_type="dataset",
)

print("Train and test datasets uploaded to Hugging Face successfully.")

Overwriting tourism_project/model_building/prep.py


###  Criteria 3
### Model Building with Experimentation Tracking

- Load the train and test data from the Hugging Face data space
- Define a model and parameters  
- Tune the model with the defined parameters 
- Log all the tuned parameters 
- Evaluate the model performance 
- Register the best model in the Hugging Face model hub

* The ML models to be built can be any of the following algorithms, such as Decision Tree, Bagging, Random Forest, AdaBoost, Gradient Boosting, and XGBoost

In [14]:
%%writefile tourism_project/model_building/train.py
import os
import pandas as pd
import mlflow
import mlflow.sklearn
import joblib
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from huggingface_hub import HfApi, create_repo
from huggingface_hub.utils import RepositoryNotFoundError

# ── Load Train / Test Data from Hugging Face ───────────────────────────────────

TRAIN_PATH = "hf://datasets/nikhileshmehta89/tourism_package_prediction/train.csv"
TEST_PATH  = "hf://datasets/nikhileshmehta89/tourism_package_prediction/test.csv"

train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)
print(f"Train shape: {train_df.shape}, Test shape: {test_df.shape}")

X_train = train_df.drop(columns=["ProdTaken"])
y_train = train_df["ProdTaken"]

X_test  = test_df.drop(columns=["ProdTaken"])
y_test  = test_df["ProdTaken"]

# ── MLflow Experiment Setup ────────────────────────────────────────────────────

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("Tourism_Package_Prediction")

# ── Define Models and Parameter Grids ─────────────────────────────────────────

models = {
    "DecisionTree": (
        DecisionTreeClassifier(random_state=42),
        {
            "max_depth": [3, 5, 10, None],
            "min_samples_split": [2, 5, 10],
            "criterion": ["gini", "entropy"],
        },
    ),
    "RandomForest": (
        RandomForestClassifier(random_state=42),
        {
            "n_estimators": [50, 100, 200],
            "max_depth": [5, 10, None],
            "min_samples_split": [2, 5],
        },
    ),
    "GradientBoosting": (
        GradientBoostingClassifier(random_state=42),
        {
            "n_estimators": [50, 100],
            "learning_rate": [0.05, 0.1, 0.2],
            "max_depth": [3, 5],
        },
    ),
}

# ── Train, Tune, and Log Each Model ───────────────────────────────────────────

best_model = None
best_model_name = ""
best_f1 = 0.0

for model_name, (estimator, param_grid) in models.items():
    print(f"\nTuning {model_name}...")

    grid_search = GridSearchCV(
        estimator=estimator,
        param_grid=param_grid,
        cv=5,
        scoring="f1",
        n_jobs=-1,
        verbose=0,
    )
    grid_search.fit(X_train, y_train)

    best_estimator = grid_search.best_estimator_
    best_params    = grid_search.best_params_

    y_pred = best_estimator.predict(X_test)
    y_prob = best_estimator.predict_proba(X_test)[:, 1]

    metrics = {
        "accuracy":  accuracy_score(y_test, y_pred),
        "f1_score":  f1_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall":    recall_score(y_test, y_pred),
        "roc_auc":   roc_auc_score(y_test, y_prob),
    }

    with mlflow.start_run(run_name=model_name):
        mlflow.log_params(best_params)
        mlflow.log_metrics(metrics)
        mlflow.sklearn.log_model(best_estimator, artifact_path="model")

    print(f"  Best params : {best_params}")
    print(f"  F1 Score    : {metrics['f1_score']:.4f}")
    print(f"  ROC-AUC     : {metrics['roc_auc']:.4f}")

    if metrics["f1_score"] > best_f1:
        best_f1 = metrics["f1_score"]
        best_model = best_estimator
        best_model_name = model_name

print(f"\nBest model: {best_model_name} with F1 = {best_f1:.4f}")

# ── Save Best Model Locally ────────────────────────────────────────────────────

os.makedirs("tourism_project/model_building", exist_ok=True)
model_path = "tourism_project/model_building/best_model.pkl"
joblib.dump(best_model, model_path)
print(f"Best model saved to {model_path}")

# ── Register Best Model to Hugging Face Model Hub ─────────────────────────────

MODEL_REPO_ID = "nikhileshmehta89/tourism-best-model"
api = HfApi(token=os.getenv("HF_TOKEN"))

try:
    api.repo_info(repo_id=MODEL_REPO_ID, repo_type="model")
    print(f"Model repo '{MODEL_REPO_ID}' already exists.")
except RepositoryNotFoundError:
    create_repo(repo_id=MODEL_REPO_ID, repo_type="model", private=False)
    print(f"Model repo '{MODEL_REPO_ID}' created.")

api.upload_file(
    path_or_fileobj=model_path,
    path_in_repo="best_model.pkl",
    repo_id=MODEL_REPO_ID,
    repo_type="model",
)
print(f"Best model ({best_model_name}) uploaded to Hugging Face model hub.")

Writing tourism_project/model_building/train.py


###  Criteria 4
### Model Deployment

- Define a Dockerfile and list all configurations
- Load the saved model from the Hugging Face model hub
- Get the inputs and save them into a dataframe
- Define a dependencies file for the deployment 
- Define a hosting script that can push all the deployment files into the Hugging Face space

## Dockerfile

In [15]:
os.makedirs("tourism_project/deployment", exist_ok=True)

In [16]:
%%writefile tourism_project/deployment/Dockerfile
# Use a minimal base image with Python 3.9 installed
FROM python:3.9

# Set the working directory inside the container to /app
WORKDIR /app

# Copy all files from the current directory on the host to the container's /app directory
COPY . .

# Install Python dependencies listed in requirements.txt
RUN pip3 install -r requirements.txt

RUN useradd -m -u 1000 user
USER user
ENV HOME=/home/user \
	PATH=/home/user/.local/bin:$PATH

WORKDIR $HOME/app

COPY --chown=user . $HOME/app

# Define the command to run the Streamlit app on port "8501" and make it accessible externally
CMD ["streamlit", "run", "app.py", "--server.port=8501", "--server.address=0.0.0.0", "--server.enableXsrfProtection=false"]

Writing tourism_project/deployment/Dockerfile


## Streamlit App

Please ensure that the web app script is named `app.py`.

In [17]:
%%writefile tourism_project/deployment/app.py
import streamlit as st
import pandas as pd
import joblib
import os
from huggingface_hub import hf_hub_download

# ── Load Model from Hugging Face Model Hub ─────────────────────────────────────

@st.cache_resource
def load_model():
    model_path = hf_hub_download(
        repo_id="nikhileshmehta89/tourism-best-model",
        filename="best_model.pkl",
        token=os.getenv("HF_TOKEN"),
    )
    return joblib.load(model_path)

model = load_model()

# ── Streamlit UI ───────────────────────────────────────────────────────────────

st.title("Tourism Wellness Package - Purchase Predictor")
st.markdown("Fill in the customer details below to predict if they will purchase the package.")

col1, col2 = st.columns(2)

with col1:
    age                     = st.number_input("Age", min_value=18, max_value=100, value=35)
    type_of_contact         = st.selectbox("Type of Contact", [0, 1], format_func=lambda x: "Company Invited" if x == 0 else "Self Enquiry")
    city_tier               = st.selectbox("City Tier", [1, 2, 3])
    duration_of_pitch       = st.number_input("Duration of Pitch (mins)", min_value=0, max_value=100, value=15)
    occupation              = st.selectbox("Occupation", [0, 1, 2, 3], format_func=lambda x: ["Free Lancer", "Large Business", "Salaried", "Small Business"][x])
    gender                  = st.selectbox("Gender", [0, 1], format_func=lambda x: "Female" if x == 0 else "Male")
    number_of_person        = st.number_input("Number of Persons Visiting", min_value=1, max_value=10, value=2)
    number_of_followups     = st.number_input("Number of Followups", min_value=1, max_value=10, value=3)

with col2:
    product_pitched         = st.selectbox("Product Pitched", [0, 1, 2, 3, 4], format_func=lambda x: ["Basic", "Deluxe", "King", "Standard", "Super Deluxe"][x])
    preferred_property_star = st.selectbox("Preferred Property Star", [3, 4, 5])
    marital_status          = st.selectbox("Marital Status", [0, 1, 2], format_func=lambda x: ["Divorced", "Married", "Single"][x])
    number_of_trips         = st.number_input("Number of Trips per Year", min_value=1, max_value=20, value=3)
    passport                = st.selectbox("Passport", [0, 1], format_func=lambda x: "No" if x == 0 else "Yes")
    pitch_satisfaction      = st.slider("Pitch Satisfaction Score", 1, 5, 3)
    own_car                 = st.selectbox("Owns a Car", [0, 1], format_func=lambda x: "No" if x == 0 else "Yes")
    children_visiting       = st.number_input("Number of Children Visiting (< 5 yrs)", min_value=0, max_value=5, value=0)
    designation             = st.selectbox("Designation", [0, 1, 2, 3, 4], format_func=lambda x: ["AVP", "Executive", "Manager", "Senior Manager", "VP"][x])
    monthly_income          = st.number_input("Monthly Income (₹)", min_value=1000, max_value=100000, value=25000)

# ── Build Input DataFrame ──────────────────────────────────────────────────────

input_data = pd.DataFrame([{
    "Age":                      age,
    "TypeofContact":            type_of_contact,
    "CityTier":                 city_tier,
    "DurationOfPitch":          duration_of_pitch,
    "Occupation":               occupation,
    "Gender":                   gender,
    "NumberOfPersonVisiting":   number_of_person,
    "NumberOfFollowups":        number_of_followups,
    "ProductPitched":           product_pitched,
    "PreferredPropertyStar":    preferred_property_star,
    "MaritalStatus":            marital_status,
    "NumberOfTrips":            number_of_trips,
    "Passport":                 passport,
    "PitchSatisfactionScore":   pitch_satisfaction,
    "OwnCar":                   own_car,
    "NumberOfChildrenVisiting": children_visiting,
    "Designation":              designation,
    "MonthlyIncome":            monthly_income,
}])

# ── Predict ────────────────────────────────────────────────────────────────────

if st.button("Predict"):
    prediction = model.predict(input_data)[0]
    probability = model.predict_proba(input_data)[0][1]

    if prediction == 1:
        st.success(f"This customer is likely to purchase the package. (Confidence: {probability:.1%})")
    else:
        st.warning(f"This customer is unlikely to purchase the package. (Confidence: {1 - probability:.1%})")

Writing tourism_project/deployment/app.py


## Dependency Handling

Please ensure that the dependency handling file is named `requirements.txt`.

In [18]:
%%writefile tourism_project/deployment/requirements.txt
streamlit==1.33.0
pandas==2.1.4
scikit-learn==1.4.2
joblib==1.3.2
huggingface_hub==0.22.2

Writing tourism_project/deployment/requirements.txt


# Hosting

In [19]:
%%writefile tourism_project/deployment/host.py
import os
from huggingface_hub import HfApi, create_repo
from huggingface_hub.utils import RepositoryNotFoundError

SPACE_REPO_ID = "nikhileshmehta89/tourism-package-predictor"
api = HfApi(token=os.getenv("HF_TOKEN"))

# Create the HF Space if it doesn't exist
try:
    api.repo_info(repo_id=SPACE_REPO_ID, repo_type="space")
    print(f"Space '{SPACE_REPO_ID}' already exists.")
except RepositoryNotFoundError:
    create_repo(
        repo_id=SPACE_REPO_ID,
        repo_type="space",
        space_sdk="streamlit",
        private=False,
    )
    print(f"Space '{SPACE_REPO_ID}' created.")

# Upload all deployment files to the HF Space
api.upload_folder(
    folder_path="tourism_project/deployment",
    repo_id=SPACE_REPO_ID,
    repo_type="space",
)
print(f"All deployment files uploaded to https://huggingface.co/spaces/{SPACE_REPO_ID}")

Writing tourism_project/deployment/host.py


###  Criteria 5
### MLOps Pipeline with Github Actions Workflow

- Create a pipeline.yml file in the GitHub repo
- Define a YAML file and list all steps to execute each step of Machine Learning
- Push all files to GitHub
- Automate the end-to-end workflow
- Update the workflow to automatically push code updates to the main branch

**Note:**

1. Before running the file below, make sure to add the HF_TOKEN to your GitHub secrets to enable authentication between GitHub and Hugging Face.
2. The below code is for a sample YAML file that can be updated as required to meet the requirements of this project.

```
name: Tourism Project Pipeline

on:
  push:
    branches:
      - main  # Automatically triggers on push to the main branch

jobs:

  register-dataset:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - name: Install Dependencies
        run: <add_code_here>
      - name: Upload Dataset to Hugging Face Hub
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: <add_code_here>

  data-prep:
    needs: register-dataset
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - name: Install Dependencies
        run: <add_code_here>
      - name: Run Data Preparation
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: <add_code_here>


  model-traning:
    needs: data-prep
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - name: Install Dependencies
        run: <add_code_here>
      - name: Start MLflow Server
        run: |
          nohup mlflow ui --host 0.0.0.0 --port 5000 &  # Run MLflow UI in the background
          sleep 5  # Wait for a moment to let the server starts
      - name: Model Building
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: <add_code_here>


  deploy-hosting:
    runs-on: ubuntu-latest
    needs: [model-traning,data-prep,register-dataset]
    steps:
      - uses: actions/checkout@v3
      - name: Install Dependencies
        run: <add_code_here>
      - name: Push files to Frontend Hugging Face Space
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: <add_code_here>

```

**Note:** To use this YAML file for our use case, we need to

1. Go to the GitHub repository for the project
2. Create a folder named ***.github/workflows/***
3. In the above folder, create a file named ***pipeline.yml***
4. Copy and paste the above content for the YAML file into the ***pipeline.yml*** file

## Requirements file for the Github Actions Workflow

## Github Authentication and Push Files

* Before moving forward, we need to generate a secret token to push files directly from Colab to the GitHub repository.
* Please follow the below instructions to create the GitHub token:
    - Open your GitHub profile.
    - Click on ***Settings***.
    - Go to ***Developer Settings***.
    - Expand the ***Personal access tokens*** section and select ***Tokens (classic)***.
    - Click ***Generate new token***, then choose ***Generate new token (classic)***.
    - Add a note and select all required scopes.
    - Click ***Generate token***.
    - Copy the generated token and store it safely in a notepad.

In [ ]:
# Install Git
!apt-get install git

# Set your Git identity (replace with your details)
!git config --global user.email "<-------GitHub Email Address------->"
!git config --global user.name "<--------GitHub UserName--------->"

# Clone your GitHub repository
!git clone https://github.com/<--------GitHub UserName--------->/<--------GitHub Reponame--------->.git

# Move your folder to the repository directory
!mv /content/tourism_project/ /content/<--------GitHub Reponame--------->

In [ ]:
# Change directory to the cloned repository
%cd <--------GitHub Reponame--------->/

# Add the new folder to Git
!git add .

# Commit the changes
!git commit -m "first commit"

# Push to GitHub (you'll need your GitHub credentials; use a personal access token if 2FA enabled)
!git push https://<--------GitHub UserName--------->:<--------GitHub Token--------->@github.com/<--------GitHub UserName--------->/<--------GitHub Reponame--------->.git

###  Criteria 5
### Output Evaluation

- GitHub (link to repository, screenshot of folder structure and executed workflow)
- Streamlit on Hugging Face (link to HF space, screenshot of Streamlit app)

- GitHub (link to repository, screenshot of folder structure and executed workflow)

- Streamlit on Hugging Face (link to HF space, screenshot of Streamlit app)

<font size=6 color="navyblue">Power Ahead!</font>
___